# Cross-species functionalis/basalis markers

Genes higher in decidual than in fibroblast nuclei (**decidual-up**, read as functionalis-like) or the reverse
(**fibroblast-up**, basalis-like) in both mouse and human, plotted on every section. The contrast is the `cell_type`
label; depth in the tissue is not measured, so "functionalis" and "basalis" are an interpretation.

1. Pseudobulk DESeq2 per species, decidual vs fibroblast, blocked by mouse timepoint or human donor.
2. Strict one-to-one orthologs (Ensembl 116): a pair is kept if both species call it in the same direction and the
   pseudobulk ratio has that sign in every section (the induced mouse timepoints and both human donors).
3. Ribosomal and mitochondrial genes are dropped, and so are genes whose fold change does not survive a gene-length
   adjustment.
4. Genes are ranked by their per-nucleus AUROC in their weakest section.
5. The top genes are plotted on every section.

A minimal rewrite of the October 25 all-timepoint run (`decidual_markers/2026_10_25_conserved_alltimepoints/` in
netscratch).

**Inputs.** The two fib/dec objects of the release are read where they are, in `xmens_final`. Everything else comes
from `xmens_scripts`; relative paths start from this notebook's folder.
- One-to-one orthologs: `orthology.one_to_one()` from `xmens_scripts/orthology`, the ortholog table of record.
- Human gene spans: the Ensembl 116 gene annotation in `orthology/build/sources/`.
- Mouse gene spans: `resources/mouse_gene_spans.tsv` (Ensembl ID, symbol, end - start + 1), made on 25 October 2026
  from the gene lines of the counting reference's GTF, `reference/transgenic_mus/genes/genes.gtf`.

**Running.** Submit `functionalis_basalis_markers.sbatch` from the folder that should receive the outputs (its header
has the commands). It writes there the executed notebook, `strict_pairs.csv`, `shortlist_decidual_up.csv`,
`shortlist_fibroblast_up.csv` and `figures/`. By hand in Jupyter, run all cells; the default `OUT_DIR = None` writes
nothing.

Checked on 8 November 2026: run from this folder, it writes tables byte-identical to those of a 7 November run that
read its inputs from their old locations, and `orthology.one_to_one()` equals that run's ortholog file row for row. On
the October inputs (release 2026_09_15) the notebook reproduces the October lists exactly (365 and 143 strict pairs,
187 and 133 shortlisted, in the same order, AUROCs equal to 1e-16), and the current release gives the same ranked
lists. A run takes 6 to 8 minutes and at most 4.2 GB of memory.

In [ ]:
# Parameters; papermill -p NAME VALUE overrides them. Relative paths start from this notebook's folder.
RELEASE = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/deliverables/2026_10_30"
ORTHOLOGY = "../../orthology"
HUMAN_GENE_ANNOTATION = "../../orthology/build/sources/ensembl_116_homo_sapiens_gene_annotation.tsv.gz"
MOUSE_GENE_SPANS = "resources/mouse_gene_spans.tsv"

OUT_DIR = None  # folder for the tables and figures; None writes nothing

# Mouse timepoints used to call markers. Uninduced has only 64 decidual nuclei, so it is plotted but not used.
# The first October run used ["1.5_dpi_late", "4.5_dpi_bleeding"].
MOUSE_TIMEPOINTS = ["1.5_dpi_early", "1.5_dpi_late", "2.5_dpi", "3.5_dpi", "4.5_dpi_bleeding"]

N_PLOT = 3  # top genes per direction to plot

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from numpy.polynomial import polynomial
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from scipy.stats import rankdata

GROUP = {"decidual_cell": "decidual", "endometrial_fibroblast": "fibroblast"}
RIBOSOMAL = r"^(RP[LS]\d+[A-Z]?\d*|RPLP\d|RPL36AL)$"  # human ribosomal protein genes
HUMAN_ORDER = ["mck6_secretory", "mck5_menstrual"]  # plotting order of the human sections
SCALE_BAR_UM = 1000

sys.dont_write_bytecode = True  # keeps a __pycache__ folder out of the orthology folder
sys.path.insert(0, ORTHOLOGY)
import orthology

out = Path(OUT_DIR) if OUT_DIR else None
if out:
    (out / "figures").mkdir(parents=True)  # fails if an earlier run left its figures here, so nothing is overwritten

## Load

The fib/dec objects of the release: `X` is log1p CP10k and `layers["counts"]` holds raw counts. A section, the
`block`, is a mouse timepoint or a human donor; lanes of one section are pooled.

In [ ]:
adata_mouse = ad.read_h5ad(f"{RELEASE}/Cevrim_XMens_Only_Fib_Dec_Cells.h5ad")
adata_human = ad.read_h5ad(f"{RELEASE}/Cevrim_Human_Only_Fib_Dec_Cells.h5ad")
adata_mouse.obs["block"] = adata_mouse.obs["timepoint"].astype(str)
adata_human.obs["block"] = adata_human.obs["donor"].astype(str) + "_" + adata_human.obs["timepoint"].astype(str)

sections = {"mouse": MOUSE_TIMEPOINTS, "human": HUMAN_ORDER}
for name, adata in [("mouse", adata_mouse), ("human", adata_human)]:
    print(name, adata.shape)
    print(pd.crosstab(adata.obs["block"], adata.obs["cell_type"]), "\n")

## 1. DESeq2 per species

Pseudobulk sums of raw counts per section and cell type, design `~ block + group`. The eligibility rule and test
settings are those of the human DE fit of record. With two human donors the human fit has one residual degree of
freedom, so p-values there are fragile; that is why step 2 also asks for the same sign in every section.

In [ ]:
def pseudobulk(adata):
    """Raw counts summed per (block, cell type): pseudobulks x genes, and each pseudobulk's block and group."""
    key = adata.obs["block"] + "__" + adata.obs["cell_type"].astype(str).map(GROUP)
    keys = sorted(key.unique())
    rows = pd.Categorical(key, categories=keys).codes
    indicator = sp.csr_matrix((np.ones(adata.n_obs), (rows, np.arange(adata.n_obs))), shape=(len(keys), adata.n_obs))
    summed = indicator @ sp.csr_matrix(adata.layers["counts"], dtype=np.float64)

    counts = pd.DataFrame(np.rint(summed.toarray()).astype(np.int64), index=keys, columns=adata.var_names)
    meta = pd.DataFrame([name.split("__") for name in keys], index=keys, columns=["block", "group"])
    return counts, meta


def fit_deseq2(counts, meta):
    """Decidual vs fibroblast; direction is +1 for a decidual-up call, -1 for fibroblast-up, 0 for no call."""
    is_decidual = (meta["group"] == "decidual").to_numpy()
    cpm = counts.div(counts.sum(axis=1), axis=0) * 1e6

    # Eligible: CPM >= 1 in at least half (rounded up) of the decidual pseudobulks, or of the fibroblast ones.
    eligible = (cpm.loc[is_decidual] >= 1).sum() >= (is_decidual.sum() + 1) // 2
    eligible |= (cpm.loc[~is_decidual] >= 1).sum() >= ((~is_decidual).sum() + 1) // 2

    inference = DefaultInference(n_cpus=1)
    dds = DeseqDataSet(
        counts=counts.loc[:, eligible],
        metadata=meta,
        design="~ block + group",
        refit_cooks=True,
        inference=inference,
        quiet=True,
    )
    dds.deseq2()
    stats = DeseqStats(
        dds,
        contrast=["group", "decidual", "fibroblast"],
        independent_filter=False,
        inference=inference,
        quiet=True,
    )
    stats.summary()

    result = stats.results_df
    result["testable"] = np.isfinite(result["log2FoldChange"]) & np.isfinite(result["pvalue"])
    is_called = result["testable"] & (result["padj"].fillna(1) < 0.05) & (result["log2FoldChange"].abs() > 1)
    result["direction"] = np.sign(result["log2FoldChange"]).where(is_called, 0).astype(int)
    return result


def species_table(adata, fit_blocks):
    """DESeq2 on the marker sections, plus every section's pseudobulk log2 CPM ratio, decidual over fibroblast."""
    counts, meta = pseudobulk(adata)
    in_fit = meta["block"].isin(fit_blocks).to_numpy()
    fit = fit_deseq2(counts.loc[in_fit], meta.loc[in_fit])

    # A pseudocount of 1 CPM keeps the ratio finite for genes undetected in one group.
    cpm = counts.div(counts.sum(axis=1), axis=0) * 1e6 + 1
    ratios = {f"ratio_{block}": np.log2(cpm.loc[f"{block}__decidual"] / cpm.loc[f"{block}__fibroblast"])
              for block in sorted(meta["block"].unique())}

    table = pd.DataFrame(ratios).join(fit[["log2FoldChange", "padj", "testable", "direction"]])
    table["eligible"] = table.index.isin(fit.index)
    table["testable"] = table["testable"].fillna(False).astype(bool)
    table["direction"] = table["direction"].fillna(0).astype(int)
    return table


de_mouse = species_table(adata_mouse, sections["mouse"])
de_human = species_table(adata_human, sections["human"])
for name, table in [("mouse", de_mouse), ("human", de_human)]:
    print(f"{name}: {table['eligible'].sum():,} eligible genes, {(table['direction'] != 0).sum():,} called "
          f"({(table['direction'] > 0).sum():,} decidual-up, {(table['direction'] < 0).sum():,} fibroblast-up)")

## 2. Conserved pairs, strict rule

Conserved: called in the same direction in both species. Strict: the pseudobulk ratio also has that sign in every
marker section, so no single section carries a call.

In [ ]:
pairs = orthology.one_to_one()  # columns human, mouse; every symbol has exactly one partner

# Pairs whose two genes exist in both objects.
in_objects = pairs["mouse"].isin(adata_mouse.var_names) & pairs["human"].isin(adata_human.var_names)
pairs = pairs[in_objects]
pairs = pairs.merge(de_mouse.add_prefix("m_"), left_on="mouse", right_index=True)
pairs = pairs.merge(de_human.add_prefix("h_"), left_on="human", right_index=True)

is_testable = pairs["m_testable"] & pairs["h_testable"]
is_conserved = is_testable & (pairs["m_direction"] == pairs["h_direction"]) & (pairs["m_direction"] != 0)
pairs["direction"] = pairs["m_direction"].where(is_conserved, 0)

ratio_columns = [f"m_ratio_{block}" for block in sections["mouse"]]
ratio_columns += [f"h_ratio_{block}" for block in sections["human"]]
sign_everywhere = np.sign(pairs[ratio_columns]).eq(pairs["direction"], axis=0).all(axis=1)
pairs["strict"] = is_conserved & sign_everywhere

print(f"{len(pairs):,} ortholog pairs, {is_testable.sum():,} testable in both species")
for direction, label in [(1, "decidual-up"), (-1, "fibroblast-up")]:
    in_direction = pairs["direction"] == direction
    print(f"{label}: {in_direction.sum():,} conserved, {(in_direction & pairs['strict']).sum():,} strict")

## 3. Ribosomal, mitochondrial and gene-length filters

Ribosomal protein genes are mostly decidual-up in both species, and decidual-up genes tend to be short and
fibroblast-up genes long. Each species' fold changes are therefore adjusted by a cubic fit on log10 gene span over
all testable pairs, and a shortlisted pair needs an adjusted fold change beyond ±1 (2-fold) in both species.

In [ ]:
spans_mouse = pd.read_csv(MOUSE_GENE_SPANS, sep="\t", names=["gene_id", "symbol", "span"])
spans_human = pd.read_csv(HUMAN_GENE_ANNOTATION, sep="\t")
spans_human["span"] = spans_human["end"] - spans_human["start"] + 1

# Spans are matched to the objects' gene symbols through their Ensembl IDs.
span_by_id_mouse = spans_mouse.drop_duplicates("gene_id").set_index("gene_id")["span"]
span_by_id_human = spans_human.drop_duplicates("ensembl_gene_id").set_index("ensembl_gene_id")["span"]
pairs["m_span"] = pairs["mouse"].map(adata_mouse.var["gene_ids"].map(span_by_id_mouse))
pairs["h_span"] = pairs["human"].map(adata_human.var["gene_ids"].map(span_by_id_human))

for prefix in ["m", "h"]:
    in_fit = is_testable & pairs[f"{prefix}_span"].notna()
    coefficients = polynomial.polyfit(
        np.log10(pairs.loc[in_fit, f"{prefix}_span"]),
        pairs.loc[in_fit, f"{prefix}_log2FoldChange"],
        3,
    )
    trend = polynomial.polyval(np.log10(pairs[f"{prefix}_span"]), coefficients)
    pairs[f"{prefix}_lfc_length_adjusted"] = pairs[f"{prefix}_log2FoldChange"] - trend

is_ribosomal_or_mito = pairs["human"].str.match(RIBOSOMAL) | pairs["human"].str.startswith("MT-")
survives_length = (
    (pairs["direction"] * pairs["m_lfc_length_adjusted"] > 1)
    & (pairs["direction"] * pairs["h_lfc_length_adjusted"] > 1)
)
pairs["shortlist"] = pairs["strict"] & ~is_ribosomal_or_mito & survives_length

strict = pairs[pairs["strict"]].copy()
print(f"strict pairs: {len(strict):,}; ribosomal or mitochondrial: {(is_ribosomal_or_mito & pairs['strict']).sum()}; "
      f"shortlisted: {strict['shortlist'].sum():,}")

## 4. Rank by the weakest section

Per-nucleus AUROC on `X` within each section, decidual nuclei as positives; for fibroblast-up genes the AUROC is
flipped (1 - AUROC), so higher always means better separation. A gene's rank comes from its weakest section.

In [ ]:
def auroc(adata, block, genes):
    """AUROC of each gene for decidual vs fibroblast nuclei of one section; tied values count half."""
    in_block = (adata.obs["block"] == block).to_numpy()
    values = adata[in_block, genes].X
    values = values.toarray() if sp.issparse(values) else np.asarray(values)
    is_decidual = (adata.obs.loc[in_block, "cell_type"] == "decidual_cell").to_numpy()
    n_decidual, n_fibroblast = is_decidual.sum(), (~is_decidual).sum()

    ranks = rankdata(values, axis=0)
    return (ranks[is_decidual].sum(axis=0) - n_decidual * (n_decidual + 1) / 2) / (n_decidual * n_fibroblast)


auroc_columns = []
for prefix, species, adata in [("m", "mouse", adata_mouse), ("h", "human", adata_human)]:
    for block in sections[species]:
        column = f"{prefix}_auroc_{block}"
        values = auroc(adata, block, strict[species].tolist())
        strict[column] = np.where(strict["direction"] > 0, values, 1 - values)
        auroc_columns.append(column)

strict["weakest_auroc"] = strict[auroc_columns].min(axis=1)
strict["weakest_section"] = strict[auroc_columns].idxmin(axis=1).str.replace("_auroc_", ":", regex=False)

# A stable sort keeps the ortholog table's order among equal AUROCs, so the ranking is reproducible.
strict = strict.sort_values(["direction", "weakest_auroc"], ascending=[False, False], kind="mergesort")
shortlist = strict[strict["shortlist"]].copy()
shortlist.insert(0, "rank", shortlist.groupby("direction").cumcount() + 1)

In [ ]:
columns = [
    "rank", "human", "mouse", "weakest_auroc", "weakest_section",
    "m_log2FoldChange", "h_log2FoldChange", "m_padj", "h_padj", "m_lfc_length_adjusted", "h_lfc_length_adjusted",
    *auroc_columns, *ratio_columns,
]
if out:
    strict.to_csv(out / "strict_pairs.csv", index=False)
for direction, label in [(1, "decidual_up"), (-1, "fibroblast_up")]:
    in_direction = shortlist[shortlist["direction"] == direction]
    if out:
        in_direction[columns].to_csv(out / f"shortlist_{label}.csv", index=False)
    print(f"{label}: {len(in_direction)} shortlisted; top 12:", ", ".join(in_direction["human"].head(12)))

## 5. Top genes on every section

Positioned nuclei of the fib/dec object (other cell types are not drawn), without the embedding outliers. Color is
log1p CP10k from 0 to the 99th percentile of the gene's nonzero values within the species; each section has its own
zoom and a 1 mm bar. The outlier flags are `obs['embedding_outlier']`, which older releases such as 2026_09_15 lack.

In [ ]:
def draw_section(ax, adata, block, expression, vmax):
    is_drawn = (
        (adata.obs["block"] == block).to_numpy()
        & adata.obs["x_um_qc"].notna().to_numpy()
        & ~adata.obs["embedding_outlier"].to_numpy(dtype=bool)
    )
    x = adata.obs["x_um_qc"].to_numpy()[is_drawn]
    y = adata.obs["y_um_qc"].to_numpy()[is_drawn]
    values = expression[is_drawn]

    # Highest values drawn last, so expressing nuclei are not hidden under non-expressing ones.
    order = np.argsort(values, kind="stable")
    points = ax.scatter(x[order], y[order], c=values[order], s=1, cmap="viridis", vmin=0, vmax=vmax,
                        linewidths=0, rasterized=True)

    x0, y0 = x.min(), y.min() - 0.04 * np.ptp(y)
    ax.plot([x0, x0 + SCALE_BAR_UM], [y0, y0], color="white", linewidth=1.5)
    ax.set_title(block.replace("_", " "), color="white", fontsize=9)
    ax.set_aspect("equal")
    ax.axis("off")
    return points


def plot_gene(mouse_gene, human_gene, title):
    """One ortholog pair on every section: mouse timepoints in the top row, human donors below."""
    mouse_blocks = ["uninduced"] + sorted(set(adata_mouse.obs["block"]) - {"uninduced"})
    fig, axes = plt.subplots(2, len(mouse_blocks), figsize=(2.4 * len(mouse_blocks), 5.2), facecolor="black")
    for ax in axes.flat:
        ax.axis("off")

    rows = [(0, adata_mouse, mouse_gene, mouse_blocks), (1, adata_human, human_gene, HUMAN_ORDER)]
    for row, adata, gene, blocks in rows:
        expression = adata[:, gene].X
        expression = (expression.toarray() if sp.issparse(expression) else np.asarray(expression)).ravel()
        nonzero = expression[expression > 0]
        vmax = np.percentile(nonzero, 99) if len(nonzero) else 1
        for column, block in enumerate(blocks):
            points = draw_section(axes[row, column], adata, block, expression, vmax)
        colorbar = fig.colorbar(points, ax=axes[row, :], fraction=0.015, pad=0.01)
        colorbar.ax.tick_params(colors="white", labelsize=7)
        colorbar.set_label(f"{gene}, log1p CP10k", color="white", fontsize=8)

    fig.suptitle(title, color="white")
    return fig


for direction, label in [(1, "decidual-up"), (-1, "fibroblast-up")]:
    for pair in shortlist[shortlist["direction"] == direction].head(N_PLOT).itertuples():
        fig = plot_gene(pair.mouse, pair.human, f"{label} #{pair.rank}: {pair.human} / {pair.mouse}")
        if out:
            fig.savefig(out / "figures" / f"{label}_{pair.rank:03d}_{pair.human}_{pair.mouse}.png", dpi=200,
                        facecolor="black")
        plt.show()